In [0]:
# Gold layer: dim_device (Kimball device dimension, Type 1 current-state)
# Source: b2b_ecommerce.silver.logs - one row per DISTINCT non-null
#         user_agent (dims-from-silver pattern; the UAs in fact_web_logs
#         are copied verbatim from silver.logs, so the fact joins this
#         dim on the shared user_agent business key).
# Why Type 1 (KPI-driven, requirements.md "Required KPIs"):
#         KPI 9 (Device-specific Conversion Metrics: conversion by
#         desktop/mobile/tablet) needs device IDENTITY only - no KPI
#         uses device history, so SCD2 would be dead weight here.
# Grain: one row per user_agent (business key + MERGE key + join key
#         from fact_web_logs). NULL user_agent rows are excluded:
#         AUTO CDC keys must be non-null, and a NULL key would
#         produce no seed rows either way (user choice - no seed rows,
#         same as the other dims). The 9,698 NULL-UA fact rows simply
#         do not match and surface as 'unknown' device in KPI views.
# Classification (rules-based, CASE first-match-wins - pre-flight
#         verified against ALL 204 distinct non-null UAs: every UA
#         maps to exactly one class and log-grain totals reconcile
#         exactly to 528,454 = 538,152 - 9,698 nulls):
#           desktop 50 UAs / 196,774 rows, mobile 14 / 153,623,
#           bot 38 / 140,414 (26.1% of all logs), tablet 44 / 37,555,
#           other 58 / 88 (corrupted UA fragments, 0.02% of logs).
#         Rules (case-insensitive, checked in order):
#           bot     - crawler/tool markers (bot|spider|crawl|slurp|
#                     nmap|masscan|zgrab|curl|python|go-http|java|
#                     postman|okhttp|dalvik|uptimerobot|wget|
#                     http-client) - includes automated scrapers,
#                     uptime checkers, security scanners
#           tablet  - iPad or SM-X700, or Android WITHOUT 'Mobile'
#           mobile  - iPhone, Android (with Mobile), or any Mobile UA
#           desktop- Windows, Macintosh, or X11 (Linux desktop)
#           other   - unrecognizable fragments (garbage/truncated UAs)
#         Corrupted fragments classify by whatever tokens survive;
#         unrecognizable ones land in 'other'.
# Bot exclusion policy (MUST-DECIDE #4): the FACT keeps bots (gross
#         traffic, requirements ask for both gross and human KPIs);
#         is_bot = (device_type = 'bot') is the one-glance exclusion
#         flag KPI views filter on for human-device metrics. 'other'
#         is NOT is_bot - views decide whether to drop it.
# Columns: device_sk + user_agent + device_type + is_bot - the
#         KPI-trimmed set. browser/os breakdowns are one-line additions
#         later; no KPI uses them now. log_rows is deliberately NOT
#         stored - it is a property of the fact (changes every load)
#         and would break snapshot-diff idempotency if frozen in the
#         dim.
# Surrogate key: device_sk = crc32(user_agent) - deterministic,
#         always non-negative and SHORT (unsigned 32-bit; user-
#         directed crc32 convention), stable across refreshes,
#         0 collisions across all 204 UAs (pre-flight verified).
# MERGE / idempotency: AUTO CDC FROM SNAPSHOT into a Delta-backed
#         streaming table, keyed on user_agent (requirements' MERGE
#         constraint), Type 1 - an identical snapshot diffs to zero
#         events, so retries and no-op runs change nothing. A UA that
#         disappears from silver is deleted here too (snapshot diff
#         semantics) - correct for a scraped UA list that should
#         track current reality.
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT flows
#         anyway (same decision as the other dims).
# Layout: not partitioned (dims are not date-partitioned),
#         liquid-clustered by device_sk (dims cluster by their
#         surrogate key).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_LOGS = "b2b_ecommerce.silver.logs"

# Crawler / automated-tool markers, matched case-insensitively on the
# lowercased UA. No real browser UA in the data contains any of these
# tokens (pre-flight verified - desktop/mobile/tablet counts reconcile
# with the hand-checked major browsers).
BOT_PATTERN = (
    "bot|spider|crawl|slurp|nmap|masscan|zgrab|curl|python|go-http|"
    "java|postman|okhttp|dalvik|uptimerobot|wget|http-client"
)


# Silver's complete current-state UA snapshot, plus the surrogate key
# and the classification. A batch temporary view: the engine reads it
# at each pipeline run (ingestion-time snapshot versioning). No
# driver-side actions anywhere. The select doubles as the column trim.
@dp.temporary_view(name="devices_snapshot")
def devices_snapshot():
    ua = F.col("user_agent")
    device_type = (
        F.when(F.lower(ua).rlike(BOT_PATTERN), "bot")
        .when(
            ua.rlike("(?i)(ipad|sm-x700)")
            | (ua.rlike("(?i)android") & ~ua.rlike("(?i)mobile")),
            "tablet",
        )
        .when(ua.rlike("(?i)(iphone|android|mobile)"), "mobile")
        .when(ua.rlike("(?i)(windows|macintosh|x11)"), "desktop")
        .otherwise("other")
    )
    return (
        spark.read.table(SILVER_LOGS)
        .where(ua.isNotNull())
        .select("user_agent")
        .distinct()
        .withColumn("device_type", device_type)
        .withColumn("device_sk", F.crc32(ua))
        .withColumn("is_bot", F.col("device_type") == F.lit("bot"))
        .select("device_sk", "user_agent", "device_type", "is_bot")
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on the
# surrogate key (dims cluster by their surrogate key).
dp.create_streaming_table(
    name="dim_device",
    comment=(
        "Kimball device dimension, SCD Type 1 (current state): one row per "
        "distinct non-null user_agent, MERGE-loaded from the "
        "b2b_ecommerce.silver.logs current-state snapshot via AUTO CDC FROM "
        "SNAPSHOT keyed on user_agent - identical snapshots diff to zero "
        "events, so retries are idempotent; UAs removed from silver are "
        "deleted here. Rules-based first-match-wins classification (bot -> "
        "tablet -> mobile -> desktop -> other), pre-flight verified across "
        "all 204 distinct UAs. is_bot = (device_type = 'bot') is the KPI-view "
        "exclusion flag - fact_web_logs keeps bot traffic (gross) and views "
        "filter it for human-device KPIs. Deterministic surrogate key "
        "device_sk = crc32(user_agent) (unsigned 32-bit, non-negative, "
        "short, 0 collisions). NULL user_agent fact rows stay unmatched and "
        "surface as 'unknown' device in views. SCD2 deliberately not applied "
        "(no KPI needs device history); browser/os columns excluded (no KPI "
        "uses them)."
    ),
    cluster_by=["device_sk"],
)


# The auto-merge load: diff each snapshot against the last committed one
# and apply inserts/updates/deletes in place (Type 1 - no history
# columns).
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_device",
    source="devices_snapshot",  # ingestion-time snapshot versioning
    keys=["user_agent"],
    stored_as_scd_type=1,
)